In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE

def augment_multilabel_data(file_path: str, label_cols: list, categorical_feature_cols: list, min_samples: int):
    """
    读取特征数据集，先对类别型特征进行独热编码，再对多标签组合进行SMOTE数据增强。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 分离特征(X)和多列标签(y) ---
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中。请检查列表: {label_cols}")
        return None
        
    y = df[label_cols]
    X = df.drop(columns=label_cols)

    # --- 步骤 3: 对特征集X中的类别型变量进行独热编码 ---
    print("\n🔄 正在对类别型特征进行独热编码...")
    
    actual_categorical_cols = [col for col in categorical_feature_cols if col in X.columns]
    if not actual_categorical_cols:
        print("   - 未在特征集中找到指定的类别列。")
        X_encoded = X
    else:
        print(f"   - 将对以下特征列进行独热编码: {actual_categorical_cols}")
        X_encoded = pd.get_dummies(X, columns=actual_categorical_cols, prefix=actual_categorical_cols)
    
    print(f"✅ 独热编码完成。特征集新维度: {X_encoded.shape}")


    # --- 步骤 4: 创建临时的“组合标签”以便SMOTE使用 ---
    y_combined = y.astype(str).agg('_'.join, axis=1)
    
    print("\n--- 增强前各类别的样本数量 ---")
    class_counts_before = y_combined.value_counts()
    print(class_counts_before.to_string())

    # --- 步骤 5: 确定需要增强的类别和目标数量 ---
    sampling_strategy = {
        cls: min_samples for cls, count in class_counts_before.items() if count < min_samples
    }

    if not sampling_strategy:
        print("\n✅ 所有类别的样本数均已达到或超过目标数量，无需增强。")
        return pd.concat([X_encoded, y], axis=1)

    print(f"\n🔄 将对以下 {len(sampling_strategy)} 个类别的样本数量增强至 {min_samples}...")
    
    # --- 步骤 6: 应用SMOTE ---
    try:
        min_class_count = class_counts_before.min()
        k_neighbors = min(min_class_count - 1, 5)
        
        if k_neighbors < 1:
            print("❌ SMOTE执行失败: 数据集中存在只有一个样本的类别，无法进行SMOTE。")
            return None

        # 注意：这里需要导入SMOTENC，因为独热编码后的X仍然可能混合了数值和类别（0/1）特征
        # 但为了简化，我们假设独热编码后所有特征都可以被标准SMOTE处理
        # 如果遇到问题，可以切换回SMOTENC
        smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_combined_resampled = smote.fit_resample(X_encoded, y_combined)
    except Exception as e:
        print(f"\n❌ SMOTE执行失败: {e}")
        return None

    # --- 步骤 7: 将增强后的“组合标签”拆分回原始的两列 ---
    y_resampled_split = y_combined_resampled.str.split('_', expand=True)
    y_resampled_split.columns = label_cols

    # --- 步骤 8: 合并增强后的特征和标签 ---
    final_df = pd.concat([X_resampled, y_resampled_split], axis=1)

    print("\n--- 增强后各类别的样本数量 ---")
    print(final_df[label_cols].astype(str).agg('_'.join, axis=1).value_counts().to_string())

    return final_df


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    # 1. 指定您的完整特征数据集文件名
    DATASET_FILE = '../data/features/dataset_complete.csv'
    
    # 2. 指定您希望用作【模型预测目标 (y)】的列名
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 3. 指定需要进行独热编码的【类别型特征 (X)】
    #    **关键修正**：此处只应包含特征列，不应包含标签列
    CATEGORICAL_FEATURE_COLUMNS = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]

    # 4. 指定每个组合类别应有的【最少样本数】
    MIN_SAMPLES_PER_CLASS = 100

    # ======================= 配置结束 ===========================

    # 执行数据增强流程
    # **关键修正**：使用正确的变量名 `CATEGORICAL_FEATURE_COLUMNS`
    augmented_df = augment_multilabel_data(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_feature_cols=CATEGORICAL_FEATURE_COLUMNS,
        min_samples=MIN_SAMPLES_PER_CLASS
    )
    
    if augmented_df is not None:
        output_file = '../data/features/source/dataset_augmented.csv'
        augmented_df.to_csv(output_file, index=False)
        
        print(f"\n\n🎉 --- 数据增强完成 --- 🎉")
        print(f"✅ 增强后的数据集共有 {len(augmented_df)} 行。")
        print(f"✅ 已保存到新文件: '{output_file}'")
        print("\n--- 增强后数据集预览 ---")
        print(augmented_df.head().to_string())

✅ 成功加载 'dataset_complete.csv'，数据集共有 411 行, 35 列。

🔄 正在对类别型特征进行独热编码...
   - 将对以下特征列进行独热编码: ['Freq', 'SenLoc', 'OR_De_Type']
✅ 独热编码完成。特征集新维度: (411, 38)

--- 增强前各类别的样本数量 ---
DE_OR_21    60
DE_OR_7     60
FE_OR_7     36
DE_IR_21    20
DE_B_14     20
DE_OR_14    20
DE_B_7      20
DE_IR_14    20
DE_IR_7     20
DE_B_21     20
FE_OR_14    15
FE_B_7      12
FE_B_14     12
FE_B_21     12
FE_IR_7     12
FE_IR_14    12
FE_IR_21    12
FE_OR_21    12
No_IN_0      8
DE_IR_28     4
DE_B_28      4

🔄 将对以下 21 个类别的样本数量增强至 100...

--- 增强后各类别的样本数量 ---
DE_B_7      100
FE_B_7      100
FE_OR_21    100
FE_OR_14    100
FE_OR_7     100
FE_IR_21    100
FE_IR_14    100
FE_IR_7     100
FE_B_21     100
FE_B_14     100
DE_OR_21    100
DE_B_14     100
DE_OR_14    100
DE_OR_7     100
DE_IR_28    100
DE_IR_21    100
DE_IR_14    100
DE_IR_7     100
DE_B_28     100
DE_B_21     100
No_IN_0     100


🎉 --- 数据增强完成 --- 🎉
✅ 增强后的数据集共有 2100 行。
✅ 已保存到新文件: 'dataset_augmented - 1.csv'

--- 增强后数据集预览 ---
   HP   rpm    Len      Mean 

# Y1

In [5]:
import pandas as pd
from imblearn.over_sampling import SMOTENC

def augment_with_smotenc(file_path: str, label_cols: list, categorical_feature_cols: list, min_samples: int):
    """
    使用SMOTENC对混合了数值和类别特征的数据集进行增强。
    此版本【不】进行独热编码。
    """
    # --- 步骤 1: 加载数据集 ---
    try:
        df = pd.read_csv(file_path)
        print(f"✅ 成功加载 '{file_path}'，数据集共有 {df.shape[0]} 行, {df.shape[1]} 列。")
    except FileNotFoundError:
        print(f"❌ 错误：文件 '{file_path}' 未找到。请确保文件名和路径正确。")
        return None

    # --- 步骤 2: 分离特征(X)和多列标签(y) ---
    if not all(item in df.columns for item in label_cols):
        print(f"❌ 错误：您指定的某些标签列不在数据集中。请检查列表: {label_cols}")
        return None
        
    y = df[label_cols]
    X = df.drop(columns=label_cols)

    # --- 步骤 3: 创建临时的“组合标签” ---
    y_combined = y.astype(str).agg('_'.join, axis=1)
    
    print("\n--- 增强前各类别的样本数量 ---")
    class_counts_before = y_combined.value_counts()
    print(class_counts_before.to_string())

    # --- 步骤 4: 确定需要增强的类别和目标数量 ---
    sampling_strategy = {
        cls: min_samples for cls, count in class_counts_before.items() if count < min_samples
    }

    if not sampling_strategy:
        print("\n✅ 所有类别的样本数均已达到或超过目标数量，无需增强。")
        return df

    print(f"\n🔄 将对以下 {len(sampling_strategy)} 个类别的样本数量增强至 {min_samples}...")
    
    # --- 步骤 5: 应用SMOTENC ---
    try:
        # **核心步骤**: 找到类别型特征在X中的列索引位置
        actual_categorical_cols = [col for col in categorical_feature_cols if col in X.columns]
        categorical_features_indices = [X.columns.get_loc(col) for col in actual_categorical_cols]
        print(f"   - 已识别出以下 {len(categorical_features_indices)} 个类别型特征列用于SMOTENC处理:\n     {actual_categorical_cols}")

        min_class_count = class_counts_before.min()
        k_neighbors = min(min_class_count - 1, 5)
        
        if k_neighbors < 1:
            print("❌ SMOTENC执行失败: 数据集中存在只有一个样本的类别，无法进行操作。")
            return None

        smotenc = SMOTENC(categorical_features=categorical_features_indices, 
                          sampling_strategy=sampling_strategy, 
                          random_state=42, 
                          k_neighbors=k_neighbors)
                          
        X_resampled, y_combined_resampled = smotenc.fit_resample(X, y_combined)
        
    except Exception as e:
        print(f"\n❌ SMOTENC执行失败: {e}")
        return None

    # --- 步骤 6: 将增强后的“组合标签”拆分回原始的多列 ---
    y_resampled_split = y_combined_resampled.str.split('_', expand=True)
    y_resampled_split.columns = label_cols

    # --- 步骤 7: 合并增强后的特征和标签 ---
    # 将numpy数组X_resampled转回DataFrame，并保持原始列名
    final_df = pd.concat([pd.DataFrame(X_resampled, columns=X.columns), y_resampled_split], axis=1)

    print("\n--- 增强后各类别的样本数量 ---")
    print(final_df[label_cols].astype(str).agg('_'.join, axis=1).value_counts().to_string())

    return final_df


# --- 如何使用 ---
if __name__ == "__main__":
    
    # ========================= 用户配置区 =========================
    
    # 1. 指定您的完整特征数据集文件名
    DATASET_FILE = '../data/features/dataset_complete.csv'
    
    # 2. 指定您希望组合起来进行均衡的【标签列】
    #    (根据您的代码截图，我更新为了3个标签)
    LABEL_COLUMNS = ['PE', 'DeLoc', 'DeS']

    # 3. **重要**: 在这里列出【所有】应该是类别型的【特征列】
    #    这些是除了标签之外的所有文本列。SMOTENC将对这些列进行特殊处理。
    CATEGORICAL_FEATURE_COLUMNS = [
        'Freq', 
        'PE', 
        'DeLoc', 
        'SenLoc',
        'OR_De_Type'
    ]

    # 4. 指定每个组合类别应有的【最少样本数】
    MIN_SAMPLES_PER_CLASS = 100

    # ======================= 配置结束 ===========================

    # 执行数据增强流程
    augmented_df = augment_with_smotenc(
        file_path=DATASET_FILE,
        label_cols=LABEL_COLUMNS,
        categorical_feature_cols=CATEGORICAL_FEATURE_COLUMNS,
        min_samples=MIN_SAMPLES_PER_CLASS
    )
    
    if augmented_df is not None:
        output_file = '../data/features/source/dataset_augmented.csv'
        augmented_df.to_csv(output_file, index=False)
        
        print(f"\n\n🎉 --- 数据增强完成 --- 🎉")
        print(f"✅ 增强后的数据集共有 {len(augmented_df)} 行。")
        print(f"✅ 列结构与输入文件完全一致。")
        print(f"✅ 已保存到新文件: '{output_file}'")
        print("\n--- 增强后数据集预览 ---")
        print(augmented_df.head().to_string())

✅ 成功加载 'dataset_complete.csv'，数据集共有 411 行, 35 列。

--- 增强前各类别的样本数量 ---
DE_OR_21    60
DE_OR_7     60
FE_OR_7     36
DE_IR_21    20
DE_B_14     20
DE_OR_14    20
DE_B_7      20
DE_IR_14    20
DE_IR_7     20
DE_B_21     20
FE_OR_14    15
FE_B_7      12
FE_B_14     12
FE_B_21     12
FE_IR_7     12
FE_IR_14    12
FE_IR_21    12
FE_OR_21    12
No_IN_0      8
DE_IR_28     4
DE_B_28      4

🔄 将对以下 21 个类别的样本数量增强至 100...
   - 已识别出以下 3 个类别型特征列用于SMOTENC处理:
     ['Freq', 'SenLoc', 'OR_De_Type']

--- 增强后各类别的样本数量 ---
DE_B_7      100
FE_B_7      100
FE_OR_21    100
FE_OR_14    100
FE_OR_7     100
FE_IR_21    100
FE_IR_14    100
FE_IR_7     100
FE_B_21     100
FE_B_14     100
DE_OR_21    100
DE_B_14     100
DE_OR_14    100
DE_OR_7     100
DE_IR_28    100
DE_IR_21    100
DE_IR_14    100
DE_IR_7     100
DE_B_28     100
DE_B_21     100
No_IN_0     100


🎉 --- 数据增强完成 --- 🎉
✅ 增强后的数据集共有 2100 行。
✅ 列结构与输入文件完全一致。
✅ 已保存到新文件: 'dataset_augmented - 1.csv'

--- 增强后数据集预览 ---
    Freq  HP   rpm    Len OR_De_Type SenLo